In [0]:
RAW_BASE_PATH = "/Volumes/main/default/datalake_retail/raw"
BRONZE_BASE_PATH = "/Volumes/main/default/datalake_retail/bronze"

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    current_date,
    lit,
    col
)

In [0]:
sales_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{RAW_BASE_PATH}/sales_transactions/")
)

In [0]:
#Add Metadata Columns
sales_bronze = (
    sales_raw
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingestion_date", current_date())
    .withColumn("source_system", lit("retail_pos"))
)

In [0]:
#Write to Bronze Delta Table
(
    sales_bronze
    .write
    .format("delta")
    .mode("append")
    .partitionBy("ingestion_date")
    .option("mergeSchema", "true")
    .save(f"{BRONZE_BASE_PATH}/sales_transactions")
)


In [0]:
#Products – Bronze Ingestion
products_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{RAW_BASE_PATH}/products/")
)

products_bronze = (
    products_raw
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingestion_date", current_date())
    .withColumn("source_system", lit("product_master"))
)

(
    products_bronze
    .write
    .format("delta")
    .mode("append")
    .partitionBy("ingestion_date")
    .save(f"{BRONZE_BASE_PATH}/products")
)


In [0]:
#Stores – Bronze Ingestion
stores_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{RAW_BASE_PATH}/stores/")
)

stores_bronze = (
    stores_raw
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingestion_date", current_date())
    .withColumn("source_system", lit("store_master"))
)

(
    stores_bronze
    .write
    .format("delta")
    .mode("append")
    .partitionBy("ingestion_date")
    .save(f"{BRONZE_BASE_PATH}/stores")
)

In [0]:
#In Bronze, incremental = new files only

In [0]:
spark.read.format("delta").load(f"{BRONZE_BASE_PATH}/sales_transactions").count()

10000

In [0]:
spark.read.format("delta").load(f"{BRONZE_BASE_PATH}/products").count()

101

In [0]:
spark.read.format("delta").load(f"{BRONZE_BASE_PATH}/stores").count()

51